In [ ]:
!apt-get install -y ffmpeg --quiet

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
!pip install opencv-python-headless numpy --quiet

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [ ]:
import os, ast, random, csv, subprocess, shutil
import numpy as np
import cv2
from pathlib import Path


In [ ]:
SEED = 42

In [ ]:

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DRIVE_BASE  = Path("/content/drive/MyDrive/AegisSafeRoad")
PROC_NORMAL = DRIVE_BASE / "datasets" / "processed" / "normal_0"
PROC_CRASH  = DRIVE_BASE / "datasets" / "processed" / "crash_1"
CCD_ANNOT   = DRIVE_BASE / "datasets" / "raws" / "CCD" / "Crash-1500.txt"

In [ ]:
TENSORS_DIR = DRIVE_BASE / "video_tensors"
TMP_DIR     = Path("/content/tmp_aegis")

In [ ]:
for split in ["train", "val", "test"]:
    (TENSORS_DIR / split).mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
N_FRAMES  = 16
IMG_H     = 224
IMG_W     = 224
TARGET_FPS = 30


In [ ]:
print("[OK] Rutas inicializadas")
print(f"     PROC_NORMAL  : {PROC_NORMAL}")
print(f"     PROC_CRASH   : {PROC_CRASH}")
print(f"     CCD_ANNOT    : {CCD_ANNOT}")
print(f"     TENSORS_DIR  : {TENSORS_DIR}")
print(f"     TMP_DIR      : {TMP_DIR}")
print(f"     N_FRAMES     : {N_FRAMES}")
print(f"     IMG_SIZE     : {IMG_H}x{IMG_W}")
print(f"     TARGET_FPS   : {TARGET_FPS}")
print(f"     DTYPE        : float16")


[OK] Rutas inicializadas
     PROC_NORMAL  : /content/drive/MyDrive/AegisSafeRoad/datasets/processed/normal_0
     PROC_CRASH   : /content/drive/MyDrive/AegisSafeRoad/datasets/processed/crash_1
     CCD_ANNOT    : /content/drive/MyDrive/AegisSafeRoad/datasets/raws/CCD/Crash-1500.txt
     TENSORS_DIR  : /content/drive/MyDrive/AegisSafeRoad/video_tensors
     TMP_DIR      : /content/tmp_aegis
     N_FRAMES     : 16
     IMG_SIZE     : 224x224
     TARGET_FPS   : 30
     DTYPE        : float16


In [ ]:
result = subprocess.run(["ffmpeg", "-version"],
                        capture_output=True, text=True)

In [ ]:
if result.returncode == 0:
    version_line = result.stdout.split("\n")[0]
    print(f"[OK] FFmpeg disponible: {version_line}")
else:
    raise RuntimeError("[X] FFmpeg no encontrado.")


[OK] FFmpeg disponible: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [ ]:
def parse_ccd_annotations(annot_path: Path) -> dict:
    """
    Parsea Crash-1500.txt.
    Retorna { "000001": { "binlabels": [...], "crash_start": int } }
    """
    annotations = {}
    with open(annot_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            bracket_start = line.index("[")
            bracket_end   = line.index("]")
            vidname       = line[:bracket_start].rstrip(",").strip()
            binlabels_str = line[bracket_start: bracket_end + 1]
            rest          = line[bracket_end + 2:].split(",")
            binlabels     = ast.literal_eval(binlabels_str)
            timing        = rest[2].strip() if len(rest) > 2 else "Unknown"
            weather       = rest[3].strip() if len(rest) > 3 else "Unknown"
            crash_start   = next((i for i, v in enumerate(binlabels) if v == 1), None)
            annotations[vidname] = {
                "binlabels"   : binlabels,
                "crash_start" : crash_start,
                "timing"      : timing,
                "weather"     : weather,
            }
    print(f"[OK] CCD annotations: {len(annotations)} videos parseados")
    return annotations


In [ ]:
ccd_annotations = parse_ccd_annotations(CCD_ANNOT)

[OK] CCD annotations: 1500 videos parseados


In [ ]:
def sample_frames_uniform(total_frames: int, n: int = N_FRAMES) -> list:
    """Uniforme — label=0 todos los normales."""
    return np.linspace(0, total_frames - 1, n).astype(int).tolist()


In [ ]:

def sample_frames_crash_centered(total_frames: int,
                                  crash_start: int,
                                  n: int = N_FRAMES) -> list:
    """
    Centrado en crash_start — CCD Crash-1500.
    8 frames antes del impacto + 8 frames desde el impacto.
    """
    half      = n // 2
    pre_start = max(0, crash_start - half)
    pre_frames  = np.linspace(pre_start, crash_start,
                               half, endpoint=False).astype(int)
    post_end    = min(total_frames - 1, crash_start + half * 2)
    post_frames = np.linspace(crash_start, post_end, half).astype(int)
    return np.concatenate([pre_frames, post_frames]).tolist()


In [ ]:

def sample_frames_biased(total_frames: int, n: int = N_FRAMES) -> list:
    """
    Sesgado 40-100% del video — TU-DAT Positive, Challenging, Kaggle crash.
    Los crashes en dashcam ocurren típicamente en la segunda mitad del clip.
    """
    start = int(0.40 * total_frames)
    end   = total_frames - 1
    return np.linspace(start, end, n).astype(int).tolist()



In [ ]:
print("[OK] Funciones de sampling definidas")
print("     sample_frames_uniform()        → label=0 (normal)")
print("     sample_frames_crash_centered() → label=1 CCD (frame-level annotation)")
print("     sample_frames_biased()         → label=1 TU-DAT + Kaggle (40-100%)")



In [ ]:
def preprocess_video_ffmpeg(src_path: Path, dst_path: Path) -> bool:
    """
    Usa FFmpeg para:
    - Escalar a 224x224
    - Forzar 30 FPS
    - Eliminar audio
    - Normalizar codec a libx264
    Retorna True si exitoso.
    """
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src_path),
        "-vf", f"scale={IMG_W}:{IMG_H}",
        "-r", str(TARGET_FPS),
        "-an",                          # eliminar audio
        "-c:v", "libx264",
        "-crf", "18",                   # calidad alta, compresión razonable
        "-preset", "ultrafast",         # rápido en CPU
        str(dst_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0


In [ ]:
def extract_frames_tensor(video_path: Path,
                           frame_indices: list) -> np.ndarray | None:
    """
    Lee los frames indicados de un video ya preprocesado (224x224, 30FPS).
    Retorna np.ndarray (16, 3, 224, 224) float16  normalizado [0, 1].
    Retorna None si el video está corrupto o tiene muy pocos frames.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < N_FRAMES:
        cap.release()
        return None

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, min(idx, total_frames - 1))
        ret, frame = cap.read()
        if not ret:
            cap.release()
            return None

        # BGR → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # HWC → CHW,  uint8 → float16 normalizado [0, 1]
        # La normalización ImageNet (mean/std) se aplica en el DataLoader
        chw = np.transpose(frame_rgb, (2, 0, 1))          # (3, 224, 224)
        chw_f16 = (chw / 255.0).astype(np.float16)        # float16 [0,1]
        frames.append(chw_f16)

    cap.release()

    tensor = np.stack(frames, axis=0)   # (16, 3, 224, 224) float16
    return tensor

In [ ]:
print("[OK] extract_frames_tensor() definida")
print(f"     Output shape : ({N_FRAMES}, 3, {IMG_H}, {IMG_W})  float16")
print(f"     Tamaño/archivo: "
      f"{N_FRAMES * 3 * IMG_H * IMG_W * 2 / 1024**2:.2f} MB  "
      f"(float16 = 2 bytes)")
print(f"     Total estimado ~6143 videos: "
      f"{6143 * N_FRAMES * 3 * IMG_H * IMG_W * 2 / 1024**3:.1f} GB")


[OK] extract_frames_tensor() definida
     Output shape : (16, 3, 224, 224)  float16
     Tamaño/archivo: 4.59 MB  (float16 = 2 bytes)
     Total estimado ~6143 videos: 27.6 GB


In [ ]:
def build_manifest(proc_normal: Path,
                   proc_crash:  Path,
                   ccd_annotations: dict) -> list:
    """
    Construye lista de dicts con metadata por video.
    Los primeros 1500 crashes (por orden de índice) se mapean a CCD annotations.
    """
    manifest = []


    for mp4 in sorted(proc_normal.rglob("*")):
        if mp4.suffix.lower() not in [".mp4", ".mov"]:
            continue
        manifest.append({
            "path"        : mp4,
            "label"       : 0,
            "sampling"    : "uniform",
            "crash_start" : None,
            "origin"      : str(mp4),
        })


    crash_files = sorted([
        f for f in proc_crash.rglob("*")
        if f.suffix.lower() in [".mp4", ".mov"]
    ])
    ccd_keys = sorted(ccd_annotations.keys())

    for i, mp4 in enumerate(crash_files):
        if i < len(ccd_keys):
            ccd_key     = ccd_keys[i]
            crash_start = ccd_annotations[ccd_key]["crash_start"]
            sampling    = "ccd_centered" if crash_start is not None else "biased"
            manifest.append({
                "path"        : mp4,
                "label"       : 1,
                "sampling"    : sampling,
                "crash_start" : crash_start,
                "origin"      : str(mp4),
            })
        else:
            manifest.append({
                "path"        : mp4,
                "label"       : 1,
                "sampling"    : "biased",
                "crash_start" : None,
                "origin"      : str(mp4),
            })

    return manifest

In [ ]:
manifest = build_manifest(PROC_NORMAL, PROC_CRASH, ccd_annotations)


In [ ]:
n_normal   = sum(1 for m in manifest if m["label"] == 0)
n_crash    = sum(1 for m in manifest if m["label"] == 1)
n_uniform  = sum(1 for m in manifest if m["sampling"] == "uniform")
n_centered = sum(1 for m in manifest if m["sampling"] == "ccd_centered")
n_biased   = sum(1 for m in manifest if m["sampling"] == "biased")
n_mov      = sum(1 for m in manifest if m["path"].suffix.lower() == ".mov")
n_mp4      = sum(1 for m in manifest if m["path"].suffix.lower() == ".mp4")


In [ ]:
print(f"\n[OK] Manifest Built: {len(manifest)} videos")
print(f"     label=0 normal       : {n_normal}")
print(f"     label=1 crash        : {n_crash}")
print(f"     .mp4                 : {n_mp4}")
print(f"     .mov                 : {n_mov}")
print(f"     sampling uniform     : {n_uniform}")
print(f"     sampling ccd_centered: {n_centered}")
print(f"     sampling biased      : {n_biased}")


[OK] Manifest Built: 6160 videos
     label=0 normal       : 3083
     label=1 crash        : 3077
     .mp4                 : 6160
     .mov                 : 0
     sampling uniform     : 3083
     sampling ccd_centered: 1500
     sampling biased      : 1577


In [ ]:
def stratified_split(manifest: list,
                     n_train: int = 5040,
                     n_val:   int = 1000,
                     n_test:  int = 120,
                     seed:    int = SEED) -> tuple:
    random.seed(seed)

    normals = [m for m in manifest if m["label"] == 0]
    crashes = [m for m in manifest if m["label"] == 1]
    random.shuffle(normals)
    random.shuffle(crashes)

    splits_def = {
        "train": (n_train // 2, n_train // 2),
        "val"  : (n_val   // 2, n_val   // 2),
        "test" : (n_test  // 2, n_test  // 2),
    }

    result = {}
    n_idx = c_idx = 0

    for name, (nn, nc) in splits_def.items():
        batch = normals[n_idx:n_idx+nn] + crashes[c_idx:c_idx+nc]
        random.shuffle(batch)
        result[name] = batch
        n_idx += nn
        c_idx += nc

    return result["train"], result["val"], result["test"]


In [ ]:
train_set, val_set, test_set = stratified_split(manifest)


In [ ]:
print(f"\n[OK] Split estratificado:")
for name, s in [("train", train_set), ("val", val_set), ("test", test_set)]:
    nn = sum(1 for m in s if m["label"] == 0)
    nc = sum(1 for m in s if m["label"] == 1)
    print(f"     {name:5s}: {len(s):4d} videos  "
          f"(normal={nn}, crash={nc}, ratio={nn/max(nc,1):.3f})")



[OK] Split estratificado:
     train: 5040 videos  (normal=2520, crash=2520, ratio=1.000)
     val  : 1000 videos  (normal=500, crash=500, ratio=1.000)
     test :  117 videos  (normal=60, crash=57, ratio=1.053)


In [ ]:
def process_split(split_name: str,
                  split_data: list,
                  tensors_dir: Path,
                  tmp_dir: Path) -> dict:
    """
    Para cada video del split:
    1. FFmpeg: estandariza a 224x224, 30FPS, sin audio → /content/tmp/
    2. OpenCV: lee N_FRAMES según estrategia de sampling
    3. Guarda tensor (16, 3, 224, 224) float16 como .npy en Drive
    4. Limpia tmp
    """
    out_dir = tensors_dir / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    stats    = {"ok": 0, "skipped": 0, "ffmpeg_err": 0, "frame_err": 0}
    manifest_rows = []
    total    = len(split_data)

    # Contadores globales por label para nombrar archivos
    counters = {0: 1, 1: 1}
    prefix   = {0: "normal", 1: "crash"}

    for i, item in enumerate(split_data):
        src_path    = item["path"]
        label       = item["label"]
        sampling    = item["sampling"]
        crash_start = item["crash_start"]

        # Nombre estandarizado del tensor
        idx      = counters[label]
        npy_name = f"{prefix[label]}_{idx:04d}.npy"
        out_npy  = out_dir / npy_name

        # Idempotente
        if out_npy.exists():
            stats["skipped"] += 1
            counters[label] += 1
            manifest_rows.append({
                "file_path"         : str(out_npy.relative_to(tensors_dir)),
                "label"             : label,
                "split"             : split_name,
                "sampling"          : sampling,
                "source_origin_path": str(src_path),
            })
            continue


        tmp_video = tmp_dir / f"tmp_{split_name}_{i}.mp4"
        ok = preprocess_video_ffmpeg(src_path, tmp_video)

        if not ok or not tmp_video.exists():
            stats["ffmpeg_err"] += 1
            print(f"  [WARN] FFmpeg falló: {src_path.name}")
            continue


        cap = cv2.VideoCapture(str(tmp_video))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        if total_frames < N_FRAMES:
            stats["frame_err"] += 1
            tmp_video.unlink(missing_ok=True)
            print(f"  [WARN] Video muy corto ({total_frames} frames): {src_path.name}")
            continue


        if sampling == "uniform":
            indices = sample_frames_uniform(total_frames)
        elif sampling == "ccd_centered" and crash_start is not None:
            indices = sample_frames_crash_centered(total_frames, crash_start)
        else:
            indices = sample_frames_biased(total_frames)

        tensor = extract_frames_tensor(tmp_video, indices)
        tmp_video.unlink(missing_ok=True)   # limpiar tmp inmediatamente

        if tensor is None:
            stats["frame_err"] += 1
            print(f"  [WARN] Error extrayendo frames: {src_path.name}")
            continue


        np.save(str(out_npy), tensor)
        stats["ok"] += 1
        counters[label] += 1

        manifest_rows.append({
            "file_path"         : str(out_npy.relative_to(tensors_dir)),
            "label"             : label,
            "split"             : split_name,
            "sampling"          : sampling,
            "source_origin_path": str(src_path),
        })

        # Progress log  /100
        if (i + 1) % 100 == 0:
            pct = (i + 1) / total * 100
            print(f"  [{split_name}] {i+1:4d}/{tota videosl} ({pct:.0f}%) — "
                  f"ok={stats['ok']}  skip={stats['skipped']}  "
                  f"err={stats['ffmpeg_err'] + stats['frame_err']}")

    print(f"\n  [{split_name}]  "
          f"ok={stats['ok']}  skip={stats['skipped']}  "
          f"ffmpeg_err={stats['ffmpeg_err']}  frame_err={stats['frame_err']}")

    return stats, manifest_rows



In [ ]:
print("\n" + "="*60)
print("  ETL Step 3 — Video → Tensor Extraction")
print("  Runtime: CPU  ")
print("="*60)

all_stats    = {}
all_manifest = []

for split_name, split_data in [("train", train_set),
                                ("val",   val_set),
                                ("test",  test_set)]:
    print(f"\n[→] Procesando: {split_name} ({len(split_data)} videos)...")
    stats, rows = process_split(
        split_name, split_data, TENSORS_DIR, TMP_DIR
    )
    all_stats[split_name] = stats
    all_manifest.extend(rows)


  ETL Step 3 — Video → Tensor Extraction
  Runtime: CPU  (~3-4 horas para 6143 videos)

[→] Procesando: train (5040 videos)...
  [train]  100/5040 (2%) — ok=100  skip=0  err=0
  [train]  200/5040 (4%) — ok=200  skip=0  err=0
  [train]  300/5040 (6%) — ok=300  skip=0  err=0
  [train]  400/5040 (8%) — ok=400  skip=0  err=0
  [train]  500/5040 (10%) — ok=500  skip=0  err=0
  [train]  600/5040 (12%) — ok=600  skip=0  err=0
  [train]  700/5040 (14%) — ok=700  skip=0  err=0
  [train]  800/5040 (16%) — ok=800  skip=0  err=0
  [train]  900/5040 (18%) — ok=900  skip=0  err=0
  [train] 1000/5040 (20%) — ok=1000  skip=0  err=0
  [train] 1100/5040 (22%) — ok=1100  skip=0  err=0
  [train] 1200/5040 (24%) — ok=1200  skip=0  err=0
  [train] 1300/5040 (26%) — ok=1300  skip=0  err=0
  [train] 1400/5040 (28%) — ok=1400  skip=0  err=0
  [train] 1500/5040 (30%) — ok=1500  skip=0  err=0
  [train] 1600/5040 (32%) — ok=1600  skip=0  err=0
  [train] 1700/5040 (34%) — ok=1700  skip=0  err=0
  [train] 1800/504

In [ ]:
manifest_path = TENSORS_DIR / "manifest.csv"
fieldnames = ["file_path", "label", "split", "sampling", "source_origin_path"]


In [ ]:
with open(manifest_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_manifest)


In [ ]:
print(f"\n[OK] manifest.csv guardado: {manifest_path}")
print(f"     Total filas: {len(all_manifest)}")



[OK] manifest.csv guardado: /content/drive/MyDrive/AegisSafeRoad/video_tensors/manifest.csv
     Total filas: 6156


In [ ]:
print("\n" + "="*60)
print("  REPORTE FINAL — ETL Step 3")
print("="*60)

total_ok  = sum(s["ok"]                          for s in all_stats.values())
total_err = sum(s["ffmpeg_err"] + s["frame_err"] for s in all_stats.values())
total_mb  = total_ok * N_FRAMES * 3 * IMG_H * IMG_W * 2 / 1024**2

for split_name, s in all_stats.items():
    split_dir = TENSORS_DIR / split_name
    npys = list(split_dir.glob("*.npy"))
    print(f"\n  [{split_name}]")
    print(f"    Extraídos : {s['ok']}")
    print(f"    Saltados  : {s['skipped']}")
    print(f"    Errores   : {s['ffmpeg_err'] + s['frame_err']}")
    print(f"    .npy en disco: {len(npys)}")

    # Verificar shape de un sample aleatorio
    if npys:
        import random as _r
        sample = np.load(str(_r.choice(npys)))
        print(f"    Sample shape : {sample.shape}  dtype={sample.dtype}")

print(f"\n  TOTAL extraídos : {total_ok}")
print(f"  TOTAL errores   : {total_err}")
print(f"  Tamaño en disco : ~{total_mb/1024:.1f} GB  (float16)")
print(f"  manifest.csv    : {len(all_manifest)} filas")


  REPORTE FINAL — ETL Step 3

  [train]
    Extraídos : 5039
    Saltados  : 0
    Errores   : 1
    .npy en disco: 5039


In [ ]:
print("\n[→] Comprimiendo video_tensors/ → video_tensors.zip ...")
print("    Esto puede tardar 20-40 minutos (operación sobre Drive).")
print("    Puedes ejecutar esta celda por separado si prefieres.")



[→] Comprimiendo video_tensors/ → video_tensors.zip ...
    Esto puede tardar 20-40 minutos (operación sobre Drive).
    Puedes ejecutar esta celda por separado si prefieres.


In [ ]:
ZIP_PATH = DRIVE_BASE / "video_tensors.tar"

result = subprocess.run(
    ["tar", "-cf", str(ZIP_PATH), str(TENSORS_DIR.name)],
    capture_output=True,
    text=True,
    cwd=str(DRIVE_BASE)
)

In [ ]:
if result.returncode == 0:
    zip_size = ZIP_PATH.stat().st_size / 1024**3
    print(f"[OK] video_tensors.zip creado: {zip_size:.2f} GB")
else:
    print(f"[WARN] Error en zip: {result.stderr[:300]}")
    print("       Puedes comprimir manualmente desde Drive.")


In [ ]:
shutil.rmtree(TMP_DIR, ignore_errors=True)
print("\n[OK] Directorio temporal /content/tmp_aegis eliminado.")